In [1]:
import os
from pathlib import Path

import sqlglot
from sqlglot import exp, expressions

from src.utils.file_utils import parse_file_name
from src.migration.com_decomposer import ComSqlDecomposer, ComDecomposerWriter
from src.migration.com_metadata import ComMetadataProcessor
from src.migration.generator import PySparkGenerator
from src.paths import *

In [2]:
USERNAME

'dungp'

In [3]:
from src.utils.source_rule_loader import load_all_source_rules

# input_file = PROJECT_ROOT / "docs" / "datalake_old" /"dml" / "com_r_k2_cif_alias.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "com" /  "com_r_mhbos_m_client_crs.sql"
input_file = DATALAKE_SCRIPT_DIR /"dml" / "com" /  "com_t_mhbos_m_client.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_contact.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_risk_profile.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer_employment.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer_employment.sql"
file_name = os.path.basename(input_file).replace('.sql', '')
layer, sub_layer, source_name, base_table = parse_file_name(input_file)
output_root = PROJECT_ROOT / "output" / "migration"

all_source_rules = load_all_source_rules()
source_rules = load_all_source_rules()[source_name] if source_name in all_source_rules else all_source_rules['default']

print(f"File gốc tại: {input_file}")

File gốc tại: C:\Users\dungp\projects\datalake-script\dml\com\com_t_mhbos_m_client.sql


In [4]:
# def run_migration_pipeline():
# ==========================================
# BƯỚC 1: BÓC TÁCH SQL (DECOMPOSER)
# ==========================================
decomposer = ComSqlDecomposer(source_rules)
decomposed_script = decomposer.decompose(input_file)

# Ghi file sub-SQL ra ổ đĩa
writer = ComDecomposerWriter()
writer.write(decomposed_script, output_root / file_name)
print(f"✅ [Bước 1] Đã bóc tách thành các block tại: {output_root / file_name / 'processing_steps'}")

# ==========================================
# BƯỚC 2: XỬ LÝ METADATA (PROCESSOR)
# ==========================================
processor = ComMetadataProcessor(source_rules)

try:
    # Hàm này sẽ phân tích AST, tự động tìm file DDL và trích xuất Schema
    pipeline_config = processor.process(decomposed_script, input_file)

    # Ghi file YAML
    metadata_output_dir = output_root / file_name / "metadata"
    processor.write_yaml(pipeline_config, metadata_output_dir)

    print(f"✅ [Bước 2] Đã xử lý Metadata thành công!")
    print(f"   -> Model nhận diện được: Model {pipeline_config['model_type']}")
    print(f"   -> Khóa (Key) nhận diện được: {pipeline_config['primary_key']['logical_primary_key']}")
    print(f"   -> File YAML đã lưu tại: {metadata_output_dir / (file_name + '.yaml')}")

except FileNotFoundError as e:
    print(f"❌ [Lỗi Bước 2]: {e}")
    print("💡 Gợi ý: Hãy đảm bảo bạn có file DDL tương ứng tại `docs/datalake_old/ddl/com_r_k2_cif_alias.sql` hoặc cùng thư mục `dml/` để hàm trích xuất Schema hoạt động!")


✅ [Bước 1] Đã bóc tách thành các block tại: C:\Users\dungp\projects\hql_spark_bridge\output\migration\com_t_mhbos_m_client\processing_steps
✅ [Bước 2] Đã xử lý Metadata thành công!
   -> Model nhận diện được: Model 3a
   -> Khóa (Key) nhận diện được: ['client_no']
   -> File YAML đã lưu tại: C:\Users\dungp\projects\hql_spark_bridge\output\migration\com_t_mhbos_m_client\metadata\com_t_mhbos_m_client.yaml


In [5]:
print("==========================================")
print(" BƯỚC 3: SINH CODE PYSPARK (GENERATOR)")
print("==========================================")

with open(output_root / file_name / "metadata" / (file_name + '.yaml'), 'r') as f:
    pipeline_config = yaml.safe_load(f)

generator = PySparkGenerator(source_rules, output_mode="simple")
ddl_context, dml_context = generator.generate(pipeline_config, output_root / file_name)

print("🎉 Hoàn tất toàn bộ Pipeline!")

 BƯỚC 3: SINH CODE PYSPARK (GENERATOR)
Generated DDL from 'model_1/com_t_ddl.jinja' at C:\Users\dungp\projects\hql_spark_bridge\output\migration\com_t_mhbos_m_client\model_1\ddl\com_t_mhbos_m_client_com_t_ddl.sql
Generated DDL from 'model_2a/com_t_ddl.jinja' at C:\Users\dungp\projects\hql_spark_bridge\output\migration\com_t_mhbos_m_client\model_2a\ddl\com_t_mhbos_m_client_com_t_ddl.sql
Generated DDL from 'model_2b/com_t_ddl.jinja' at C:\Users\dungp\projects\hql_spark_bridge\output\migration\com_t_mhbos_m_client\model_2b\ddl\com_t_mhbos_m_client_com_t_ddl.sql
Generated DDL from 'model_3/com_t_ddl.jinja' at C:\Users\dungp\projects\hql_spark_bridge\output\migration\com_t_mhbos_m_client\model_3\ddl\com_t_mhbos_m_client_com_t_ddl.sql
Generated DDL from 'model_3a/com_m_ddl.jinja' at C:\Users\dungp\projects\hql_spark_bridge\output\migration\com_t_mhbos_m_client\model_3a\ddl\com_t_mhbos_m_client_com_m_ddl.sql
Generated DDL from 'model_3a/com_t_ddl.jinja' at C:\Users\dungp\projects\hql_spark_br

In [6]:
pipeline_config

{'pipeline_id': 'com_t_mhbos_m_client',
 'layer': 'com',
 'model_type': '3a',
 'source_name': 'mhbos',
 'target_table_name': 't_mhbos_m_client',
 'columns': [{'name': 'client_no', 'type': 'VARCHAR(9)', 'remark': None},
  {'name': 'clean_rule_flag', 'type': 'VARCHAR(60)', 'remark': None},
  {'name': 'primary_identification_type',
   'type': 'VARCHAR(10)',
   'remark': None},
  {'name': 'primary_identification_no', 'type': 'VARCHAR(60)', 'remark': None},
  {'name': 'secondary_identification_type',
   'type': 'VARCHAR(10)',
   'remark': None},
  {'name': 'secondary_identification_no',
   'type': 'VARCHAR(60)',
   'remark': None},
  {'name': 'customer_name', 'type': 'VARCHAR(250)', 'remark': None},
  {'name': 'customer_name_concatenate',
   'type': 'VARCHAR(250)',
   'remark': None},
  {'name': 'client_name', 'type': 'VARCHAR(60)', 'remark': None},
  {'name': 'client_name1', 'type': 'VARCHAR(60)', 'remark': None},
  {'name': 'client_name2', 'type': 'VARCHAR(60)', 'remark': None},
  {'name'

In [7]:
dml_context["main_processing_sqls"]

['/* 3.2 insert data to target table */\nWITH today_accounts AS (\n  SELECT DISTINCT\n    client_no\n  FROM {params["com_schema"]}.temp_t_mhbos_m_client\n)\nINSERT INTO {params["com_schema"]}.temp_t_mhbos_m_client_consolidated\nSELECT\n  client_no,\n  clean_rule_flag,\n  primary_identification_type,\n  primary_identification_no,\n  secondary_identification_type,\n  secondary_identification_no,\n  customer_name,\n  customer_name_concatenate,\n  client_name,\n  client_name1,\n  client_name2,\n  client_name3,\n  mobile_no,\n  fax_no,\n  tel_no_home,\n  tel_no_office,\n  date_of_birth,\n  race,\n  email_1,\n  email_2,\n  email_3,\n  email_4,\n  email_5,\n  email_6,\n  email_7,\n  email_8,\n  email_9,\n  email_10,\n  sex,\n  addr1,\n  addr2,\n  addr3,\n  addr4,\n  postcode,\n  city,\n  state,\n  perm_addr1,\n  perm_addr2,\n  perm_addr3,\n  perm_addr4,\n  perm_postcode,\n  perm_city,\n  perm_state,\n  noms_ind,\n  cleaned_nominees_name,\n  principal_name,\n  intermediary_name,\n  beneficiary

In [8]:

# Read pre_processing SQLs
pre_processing_sqls = []
for step in pipeline_config.get("pre_processing", []):
    if step.get("action") == "skip":
        continue
    step_file = Path(step["file"])
    if step_file.exists():
        pre_processing_sqls.append(step_file.read_text(encoding="utf-8"))